In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from algbench import read_as_pandas
from tspn_bnb2.misc.paper_style import FULLWIDEFIGURE, init_params
from tspn_bnb2.schemas import AnnotatedInstance, AnnotatedSolution

init_params()

In [ ]:
map_root_name = {
    "LongestEdgePlusFurthestSite": "LEFP",
    "RandomRoot": "Random",
    "OrderRoot": "CHR",
    "LongestPair": "LE",
}

In [ ]:
def read_row(row):
    instance = AnnotatedInstance.model_validate_json(
        row["parameters"]["args"]["kwargs"]["instance_json"]
    )
    solution = AnnotatedSolution.model_validate_json(row["result"]["solution"])
    return {
        "solution": solution,
        "upper_bound": solution.upper_bound,
        "lower_bound": solution.lower_bound,
        "is_optimal": solution.is_optimal,
        "instance_type": instance.derive_instance_type(),
        "instance_name": row["parameters"]["args"]["kwargs"]["instance_name"],
        "instance": instance,
        "solve_time": row["result"]["solve_time"],
        "n": instance.num_polygons(),
        "root_strategy": map_root_name[row["parameters"]["args"]["alg_params"]["root"]],
    }


benchmark = read_as_pandas("results_root_strategy", read_row)
benchmark = benchmark.sort_values(by=["root_strategy"])
print(
    "loaded",
    len(benchmark),
    "runs",
    benchmark["root_strategy"].unique(),
    len(benchmark["instance_name"].unique()),
)

In [ ]:
# validate that solutions are correct.
n_checks = 0
for _, row in benchmark.iterrows():
    solution = row["solution"]
    if solution is None:
        continue
    same_instance = benchmark[benchmark["instance_name"] == row["instance_name"]]

    assert solution.check_feasibility(row["instance"], eps=0.01)
    for _, other in same_instance.iterrows():
        if other["solution"] is None:
            continue
        check = solution.plausibility_check(other["solution"], eps=0.01)
        assert check["valid"], (
            f"Check failed for {row['instance_name']} {check}"
            f" {other['solution']} {row['root_strategy']}"
        )
        n_checks += 1

print(n_checks, "checks succeeded")

In [ ]:
fig, ax = plt.subplots(figsize=FULLWIDEFIGURE)

sns.boxenplot(
    benchmark,
    x="root_strategy",
    y="solve_time",
    hue="instance_type",
    hue_order=["OSM", "tessellation", "random"],
)

xticks = list(ax.get_xticks())
xticklabels = list()

for label in ax.get_xticklabels():
    solutions_for_label = benchmark[(benchmark["root_strategy"] == label.get_text())]
    optimal_percentage = len(solutions_for_label[solutions_for_label["is_optimal"]]) / len(
        solutions_for_label
    )

    label = label.get_text() + f"\n[{round(optimal_percentage * 100, 2)}\\%]"
    xticklabels.append(label)

ax.set_xticks(xticks)
ax.set_xticklabels(xticklabels)

ax.set_title("lower is better")
ax.set_xlabel("root node strategy")
ax.set_ylabel("runtime [s]")

leg = ax.legend(loc="upper right")
leg.set_title(None)

fig.savefig("../plots/rq1_root_node/runtimes.pdf", bbox_inches="tight")

In [ ]:
# adapted from OR-Tools Primer: https://d-krupke.github.io/cpsat-primer/08_benchmarking.html#performance-plots-for-solution-quality-within-a-time-limit

import numpy as np
import pandas as pd
from matplotlib.axes import Axes


def plot_performance_profile(
    data: pd.DataFrame,
    ax: Axes,
    instance_column: str,
    strategy_column: str,
    metric_column: str,
    direction: str,
    comparison: str = "relative",
    title: str = None,
    highlight_best: bool = False,
    scale: str | None = None,
    log_base: int = 2,
    eps: float = 0.01,
) -> Axes:
    """Plot a performance profile.

    Either on a relative-ratio basis or absolute-difference basis:
      - For comparison="relative":
          x-axis: performance ratio t (log scale if t_max > 10)
          t = (value / best) if direction="min",
          or t = (best / value) if direction="max".
      - For comparison="absolute":
          x-axis: absolute difference D = (value - best)
          if direction="min",
          or D = (best - value) if direction="max".
      - y-axis: proportion of problems with t (or D) <= x
        for each solver.
      - If highlight_best=True, detect and bold the
        dominating solver curve (AUC in appropriate space).
      - Ensures a reasonable number of ticks on the x-axis.

    Args:
        data: DataFrame with columns [instance, strategy, metric].
        instance_column: column identifying each problem instance.
        strategy_column: column identifying each solver/strategy.
        metric_column: column of the performance metric.
        direction: "min" if lower metric is better,
            "max" if higher is better.
        comparison: "relative" or "absolute".
        title: Optional plot title.
        highlight_best: If True, find the solver with largest
            AUC and draw it in bold.
        ax: An existing matplotlib Axes to draw into.
        scale: x-axis scale override ("linear" or "log");
            if None, chosen automatically.
        log_base: base for log scale if used (default 2).
        eps: tolerance for comparison.

    Returns:
        The matplotlib Axes containing the performance profile.

    """
    if direction not in ("min", "max"):
        raise ValueError("`direction` must be 'min' or 'max'.")
    if comparison not in ("relative", "absolute"):
        raise ValueError("`comparison` must be 'relative' or 'absolute'.")

    # 1) Compute best value per instance
    best_val = data.groupby(instance_column)[metric_column].agg(direction)

    # 2) Pivot to get per-instance x per-strategy medians
    pivot = (
        data.groupby([instance_column, strategy_column])[metric_column]
        .median()
        .unstack(fill_value=np.nan)
    )

    # 3) Build comparison matrix C[p, s]
    comp = pd.DataFrame(index=pivot.index, columns=pivot.columns, dtype=float)

    if comparison == "relative":
        best_val = best_val * (1 - eps) if direction == "min" else best_val * (1 + eps)

        for strat in pivot.columns:
            if direction == "min":
                comp[strat] = pivot[strat] / best_val
            else:  # direction == "max"
                comp[strat] = best_val / pivot[strat]

        comp = comp.replace([np.inf, -np.inf, 0.0], np.nan)

    else:  # comparison == "absolute"
        for strat in pivot.columns:
            if direction == "min":
                comp[strat] = pivot[strat] - best_val
            else:  # direction == "max"
                comp[strat] = best_val - pivot[strat]
        comp = comp.replace([np.inf, -np.inf], np.nan)

    # 4) Collect all distinct x-values, including baseline
    all_vals = comp.values.flatten()
    finite_vals = all_vals[np.isfinite(all_vals)]
    baseline = 1.0 if comparison == "relative" else 0.0
    all_x = np.unique(np.sort(finite_vals))
    all_x = np.concatenate(([baseline], all_x))
    all_x = np.unique(np.sort(all_x))

    # 5) Build performance-profile DataFrame
    n_instances = comp.shape[0]
    profile = pd.DataFrame(index=all_x, columns=comp.columns, dtype=float)

    for x in all_x:
        leq = (comp <= x).sum(axis=0)
        profile.loc[x] = leq / n_instances

    # 6) Identify dominating solver if requested (max AUC)
    best_solver = None
    if highlight_best:
        if comparison == "relative":
            log_x = np.log(all_x)
            areas = {}
            for strat in profile.columns:
                y = profile[strat].astype(float).values
                areas[strat] = np.trapz(y, x=log_x)
            best_solver = max(areas, key=areas.get)
        else:
            areas = {}
            for strat in profile.columns:
                y = profile[strat].astype(float).values
                areas[strat] = np.trapz(y, x=all_x)
            best_solver = max(areas, key=areas.get)

    fig = ax.figure

    # 8) Determine scale if not overridden
    if scale is None:
        if comparison == "relative" and all_x[-1] > 10:
            use_log = True
        else:
            use_log = False
    else:
        use_log = scale == "log"

    # 9) Plot each solver's curve
    for strat in profile.columns:
        y = profile[strat].astype(float)
        if highlight_best and strat == best_solver:
            ax.step(
                all_x,
                y,
                where="post",
                label=strat,
                linewidth=3.0,
                alpha=1.0,
            )
        else:
            ax.step(
                all_x,
                y,
                where="post",
                label=strat,
                linewidth=1.5,
                alpha=0.6 if highlight_best else 1.0,
            )

    # 10) Axis scaling and limits
    if comparison == "relative":
        if use_log:
            ax.set_xscale("log", base=log_base)
        else:
            ax.set_xscale("linear")
            ax.set_xlim(1 + eps, ax.get_xlim()[1])
        xlabel = (
            f"Within this factor of the best (log{log_base} scale)"
            if use_log
            else "Within this factor of the best (linear scale)"
        )
    else:  # absolute
        ax.set_xscale("linear")
        ax.set_xlim(0.0, all_x[-1] * 1.1)
        xlabel = "Absolute difference from the best"

    if title is not None:
        ax.set_title(title)

    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel("Proportion of problems", fontsize=12)

    ax.axvline(
        x=baseline,
        color="gray",
        linestyle="--",
        alpha=0.7,
    )
    ax.grid(True, which="both", linestyle=":", linewidth=0.5)

    # 11) Legend inside lower right
    ax.legend(loc="lower right", frameon=False)

    fig.tight_layout()
    return ax

In [ ]:
fig, axs = plt.subplots(ncols=2, nrows=1, figsize=(8, 3))

plot_performance_profile(
    benchmark,
    ax=axs[0],
    instance_column="instance_name",
    comparison="relative",
    direction="min",
    metric_column="upper_bound",
    strategy_column="root_strategy",
    title="performance profile on objective value",
)

plot_performance_profile(
    benchmark,
    ax=axs[1],
    instance_column="instance_name",
    comparison="relative",
    direction="max",
    metric_column="lower_bound",
    strategy_column="root_strategy",
    title="performance profile on lower bound",
)

# ax.set_title("lower is better")
# ax.set_xlabel("root node strategy")
# ax.set_ylabel("runtime (s)")
fig.tight_layout()